In [201]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from jax import lax, jit
from jax import config
config.update("jax_enable_x64", True)

# Pseudo-Spectral Solver for the 2D Taylor–Green Vortex

The problem is defined on a doubly periodic domain, making Fourier basis functions an ideal choice. The Fourier transform is defined as:

$$\mathcal{F}(u) = \int_{-\infty}^{\infty} u(x)\, e^{-ikx}\, dx$$

A key property that makes spectral methods attractive is the exact differentiation in frequency space:

$$\mathcal{F}\left[\frac{d^n f}{dx^n}\right] = (ik)^n\hat{f}$$

This means spatial derivatives reduce to pointwise multiplications by wavenumbers in Fourier space.
## Diffusion Term

For a purely diffusive problem (e.g. the 2D parabolic heat equation), the semi-discrete system in Fourier space becomes:

$$\frac{d\hat{\mathbf{u}}}{dt} = -\nu\,\mathbf{k}^2 \odot \hat{\mathbf{u}}$$

where $\odot$ denotes the Hadamard (element-wise) product and $\mathbf{k}^2 = k_x^2 + k_y^2$. This is a diagonal linear system — each Fourier mode evolves independently and can be integrated exactly.

## Advection Term and the Pseudo-Spectral Approach

The advection term $(\mathbf{u} \cdot \nabla)\mathbf{u}$ is nonlinear. In Fourier space, it becomes a convolution:

$$\widehat{(\mathbf{u} \cdot \nabla)\mathbf{u}}(k) = i\int_{-\infty}^{\infty} \hat{u}(k - k')\, k'\, \hat{u}(k')\, dk'$$

Evaluating this convolution directly is $\mathcal{O}(N^2)$ per mode and very expensive. Instead, the standard approach is to:

1. Transform each factor to physical space via inverse FFT
2. Multiply pointwise in physical space
3. Transform the product back to Fourier space

This reduces the cost to $\mathcal{O}(N \log N)$ via the FFT, and is precisely why the method is called **pseudo-spectral**, the nonlinear terms are evaluated in physical space rather than purely in frequency space.

## De-aliasing

Pointwise multiplication in physical space introduces aliasing errors: energy from high-wavenumber modes reflects back into resolved modes, corrupting the solution. This is suppressed by applying the **2/3 rule**, zeroing all Fourier coefficients with $|k| > \frac{2}{3}k_{\max}$ after each nonlinear evaluation.

## Time Integration

The semi-discrete vorticity equation takes the form:

$$\frac{d\hat{\omega}}{dt} = \underbrace{-\nu k^2 \hat{\omega}}_{\text{linear}} + \underbrace{\hat{N}(\hat{\omega})}_{\text{nonlinear}}$$

Two time integrators are implemented:

- **RK4** — a classical explicit scheme
- **ETDRK2** (Exponential Time Differencing RK2) — integrates the linear term $-\nu k^2$ **exactly** via the matrix exponential $e^{-\nu k^2 \Delta t}$, treating only the nonlinear part explicitly. 

In [257]:
def get_kappa(Nx, Ny, Lx, Ly):
    kappa_x = jnp.fft.fftfreq(Nx, Lx/(Nx*2.0*jnp.pi))
    kappa_y = jnp.fft.rfftfreq(Ny, Ly/(Ny*2.0*jnp.pi))

    Kappa_X, Kappa_Y = jnp.meshgrid(kappa_x, kappa_y, indexing="ij")

    return Kappa_X, Kappa_Y

def aliasing_mask(KX, KY):
    kx_max = jnp.max(jnp.abs(KX[:, 0]))
    ky_max = jnp.max(jnp.abs(KY[0, :]))

    mask = ((jnp.abs(KX) <= (2.0/3.0)*kx_max) &
            (jnp.abs(KY) <= (2.0/3.0)*ky_max))
    return mask

def Spectral_Poisson(u, b_hat, kappa_x, kappa_y):
    
    kappa = kappa_x**2 + kappa_y**2
    kappa = kappa.at[0,0].set(1.0)

    b_hat = b_hat.at[0,0].set(0.0)

    uhat_next = b_hat / kappa
    # u_next = jnp.fft.irfft2(uhat_next)

    return uhat_next

def Spectral_Laplacian(u_hat, kappa_x, kappa_y):
    u_hat_xx_yy = -(kappa_x**2 + kappa_y**2) * u_hat
    
    return u_hat_xx_yy

def Spectral_Divergence(u_hat, psi_hat, kappa_x, kappa_y, mask,):
    factor_x = (1j)*kappa_x
    factor_y = (1j)*kappa_y
    psi_x_hat = factor_x * psi_hat; psi_y_hat = factor_y * psi_hat
    vor_x_hat = factor_x * u_hat; vor_y_hat = factor_y * u_hat

    psi_x = jnp.fft.irfft2(psi_x_hat); psi_y = jnp.fft.irfft2(psi_y_hat)
    vor_x = jnp.fft.irfft2(vor_x_hat); vor_y = jnp.fft.irfft2(vor_y_hat)
    

    convection_hat = jnp.fft.rfft2(psi_y * vor_x - psi_x * vor_y) 
    convection_masked = convection_hat * mask
     
    return convection_masked
    
def Spectral_rhs(u, psi, kappa_x, kappa_y, mask, ν = 0.01):

    u_hat = jnp.fft.rfft2(u)
    # print("Uhat_1st:", u_hat.shape)
    
    b = -u_hat
    # ------------- Call Poisson Equation
    psi_hat = Spectral_Poisson(psi, b, kappa_x, kappa_y)

    # ------------- Precompute Psi_hat
    # psi_hat = jnp.fft.rfft2(psi)
    # print("Psi_1st:", psi_hat.shape)
    
    # ------------- Construct Momentum Equation rhs
    ConvectiveTerm = Spectral_Divergence(u_hat, psi_hat, kappa_x, kappa_y, mask)
    # print("Convective Term", ConvectiveTerm.shape)
    DiffusiveTerm  = Spectral_Laplacian(u_hat, kappa_x, kappa_y)
    # print("DIffusive Term", DiffusiveTerm.shape)
    rhs_hat = ν*DiffusiveTerm - ConvectiveTerm
    # print("rhs_hat", rhs_hat.shape)
    rhs = jnp.fft.irfft2(rhs_hat)
    # print("rhs", rhs.shape)

    return rhs
def rk4_step(u, dt, psi, kappa_x, kappa_y, mask,):
    k1 = Spectral_rhs(u, psi, kappa_x, kappa_y, mask,)
    k2 = Spectral_rhs(u + 0.5 * dt * k1, psi, kappa_x, kappa_y, mask)
    # print("k2", k2.shape)
    k3 = Spectral_rhs(u + 0.5 * dt * k2, psi, kappa_x, kappa_y, mask)
    # print("k3", k3.shape)
    k4 = Spectral_rhs(u + dt * k3, psi, kappa_x, kappa_y, mask)
    # print("k4", k4.shape)
    res = u + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
    # print("result", res.shape)
    return res

def ETDRK2(uhat, dt, psi_hat, kappa_x, kappa_y, mask, nu):
    c = - (kappa_x**2 + kappa_y**2) * nu
    int_factor = jnp.exp(c*dt)
    # -----------  Linear Term 
    linear_term = int_factor * uhat
    
    # -----------  Non-Linear Treatment
    int_fac_NonLinear_0 = jnp.where(
        c==0,
        dt,
        (int_factor - 1)/c
    )
    int_fac_NonLinear_1 = jnp.where(
        c==0,
        dt,
        (int_factor - 1 - dt*c) / (dt*c**2)
    )

    # Step ZERO
    F_at_0 = Spectral_Divergence(uhat, psi_hat, kappa_x, kappa_y, mask)
    nonlinear_term_0 = int_fac_NonLinear_0 *  F_at_0
    u_next_0 = linear_term + nonlinear_term_0

    # Step ONE
    psi_1 = Spectral_Poisson(psi, -u_next_0, kappa_x, kappa_y)
    psi_hat_1 = jnp.fft.rfft2(psi)
    F_at_1 = Spectral_Divergence(u_next_0, psi_hat_1, kappa_x, kappa_y, mask)
    u_next_1 = u_next_0 + (F_at_1 - F_at_0)*int_fac_NonLinear_1

    u_next_final = jnp.fft.irfft2(u_next_1)
    return u_next_final



@jit
def solve(u, dt, psi, kappa_x, kappa_y, mask, nu):
    # u_next_hat = rk4_step(u, dt, psi, kappa_x, kappa_y, mask)

    # ------------- ETDRK2 choice
    u_hat = jnp.fft.rfft2(u)    
    b = -u_hat
    # ------------- Call Poisson Equation
    psi_hat = Spectral_Poisson(psi, b, kappa_x, kappa_y)

    u_next_hat = ETDRK2(u_hat, dt, psi_hat, kappa_x, kappa_y, mask, nu)

    return u_next_hat


In [258]:
Nx = 32
Ny = 32
Lx = 2.0*jnp.pi
Ly = Lx
u0 = 1.0
ν = 0.01
t_end = 10.0
dt = 0.001
total_timesteps = int(t_end/dt)

x = jnp.linspace(0.0, Lx, Nx, endpoint=False)
y = jnp.linspace(0.0, Ly, Ny, endpoint=False)
X, Y = jnp.meshgrid(x,y)

# ----- Initialise Fields 
omega = jnp.zeros((Nx, Ny))
psi = jnp.zeros_like(omega)

omega = 2.0*u0*jnp.cos(X)*jnp.cos(Y)
exact_sloution = 2.0*u0*jnp.exp(-2.0*ν*t_end)*jnp.cos(X)*jnp.cos(Y)
fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize=(12,5))
ax1 = axes[0]
im1 = ax1.contourf(X, Y, omega,cmap='plasma')
ax1.set_title('Initial Vorticity Distri')
ax1.set_xlabel('x')
ax1.set_ylabel('y')


kappa_x, kappa_y = get_kappa(Nx, Ny, Lx, Ly)
mask = aliasing_mask(kappa_x, kappa_y)
timestep_init = 0
state = (omega, dt, psi, kappa_x, kappa_y, mask, timestep_init)
# omega_new, _, psi_new, _, _, _, timestep_final = lax.while_loop(cond_fn, body_fn, state)

while timestep_init < total_timesteps:
    omega = solve(omega, dt, psi, kappa_x, kappa_y, mask, ν)
    timestep_init = timestep_init+1

# ax2 = axes[1]
# im2 = ax2.contourf(X, Y, omega, cmap='plasma')
# plt.show()

In [259]:
def L2(u, uref):
    return 1.0/(uref.shape[0]*uref.shape[1])*jnp.sqrt(jnp.sum((u - uref)**2))
def Linf(u, uref):
    return jnp.max(abs(uref - u))

In [260]:
print(Linf(omega, exact_sloution))
print(L2(omega, exact_sloution))

1.0473844014313727e-12
1.141366368076366e-14


In [261]:
print(Linf(omega, exact_sloution))
print(L2(omega, exact_sloution))

1.0473844014313727e-12
1.141366368076366e-14


In [262]:
print(Linf(omega, exact_sloution))
print(L2(omega, exact_sloution))

1.0473844014313727e-12
1.141366368076366e-14
